In [23]:
import pandas as pd
df = pd.read_csv("student_data.csv")

In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 395 entries, 0 to 394
Data columns (total 33 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   school      395 non-null    object
 1   sex         395 non-null    object
 2   age         395 non-null    int64 
 3   address     395 non-null    object
 4   famsize     395 non-null    object
 5   Pstatus     395 non-null    object
 6   Medu        395 non-null    int64 
 7   Fedu        395 non-null    int64 
 8   Mjob        395 non-null    object
 9   Fjob        395 non-null    object
 10  reason      395 non-null    object
 11  guardian    395 non-null    object
 12  traveltime  395 non-null    int64 
 13  studytime   395 non-null    int64 
 14  failures    395 non-null    int64 
 15  schoolsup   395 non-null    object
 16  famsup      395 non-null    object
 17  paid        395 non-null    object
 18  activities  395 non-null    object
 19  nursery     395 non-null    object
 20  higher    

In [25]:
df['risk'] = df['G3'].apply(lambda x: 1 if x < 10 else 0)

In [26]:
df['risk'].value_counts()

,count
risk,
0,265
1,130


In [27]:
X = df.drop(['G3', 'risk'], axis=1)
y = df['risk']

In [28]:
X = df.drop(['G3', 'G2', 'risk'], axis=1)
y = df['risk']

In [29]:
X = df.drop(['G3', 'G2', 'risk'], axis=1)
y = df['risk']

In [30]:
X_encoded = pd.get_dummies(X, drop_first=True)
X_encoded.shape

(395, 40)

In [31]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [32]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight='balanced'
)
model.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', random_state=42)

In [33]:
y_pred = model.predict(X_test)

In [34]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.8481012658227848

Classification Report:
               precision    recall  f1-score   support

           0       0.86      0.92      0.89        53
           1       0.82      0.69      0.75        26

    accuracy                           0.85        79
   macro avg       0.84      0.81      0.82        79
weighted avg       0.85      0.85      0.84        79


Confusion Matrix:
 [[49  4]
 [ 8 18]]


In [35]:
import numpy as np

y_probs = model.predict_proba(X_test)[:,1]

y_pred_new = np.where(y_probs > 0.4, 1, 0)

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print("Accuracy:", accuracy_score(y_test, y_pred_new))
print("\nClassification Report:\n", classification_report(y_test, y_pred_new))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_new))

Accuracy: 0.8354430379746836

Classification Report:
               precision    recall  f1-score   support

           0       0.93      0.81      0.87        53
           1       0.70      0.88      0.78        26

    accuracy                           0.84        79
   macro avg       0.82      0.85      0.82        79
weighted avg       0.86      0.84      0.84        79


Confusion Matrix:
 [[43 10]
 [ 3 23]]


In [36]:
import pandas as pd

feature_importance = pd.Series(
    model.feature_importances_,
    index=X_encoded.columns
).sort_values(ascending=False)

print(feature_importance.head(10))

G1               0.355717
absences         0.052571
failures         0.049045
age              0.033620
goout            0.032892
health           0.029043
famrel           0.029005
freetime         0.027707
Fedu             0.026138
schoolsup_yes    0.024257
dtype: float64


In [37]:
def generate_explanation(student_row):
    reasons = []

    if student_row['G1'] < 10:
        reasons.append("Low early academic performance (G1)")

    if student_row['absences'] > 10:
        reasons.append("High absenteeism")

    if student_row['failures'] > 0:
        reasons.append("History of academic failures")

    if student_row['studytime'] <= 1:
        reasons.append("Low study time")

    if student_row['goout'] >= 4:
        reasons.append("High social activity affecting academics")

    return reasons

In [38]:
risk_students = df.loc[X_test[y_pred_new == 1].index]
student = risk_students.iloc[0]

In [39]:
reasons = generate_explanation(student)
print("Identified Risk Factors:")
for r in reasons:
    print("-", r)

Identified Risk Factors:
- Low early academic performance (G1)
- History of academic failures
- Low study time


In [40]:
def generate_summary(reasons):
    if not reasons:
        return "Student shows moderate performance indicators. Monitor regularly."

    summary = "The student is at academic risk due to: "
    summary += ", ".join(reasons)
    summary += ". Early academic intervention is recommended."

    return summary

print(generate_summary(reasons))

The student is at academic risk due to: Low early academic performance (G1), History of academic failures, Low study time. Early academic intervention is recommended.


In [41]:
import pandas as pd

# Get predictions for full test set
results = X_test.copy()
results['Actual'] = y_test
results['Predicted'] = y_pred_new

# Add gender column from original df
results['sex'] = df.loc[X_test.index, 'sex']

# Risk prediction rate by gender
print(results.groupby('sex')['Predicted'].mean())

sex
F    0.441860
M    0.388889
Name: Predicted, dtype: float64


In [45]:
import joblib

joblib.dump(model, "student_risk_model.pkl")

['student_risk_model.pkl']

In [43]:
# threshold = 0.4

In [46]:
joblib.dump(X_encoded.columns.tolist(), "model_columns.pkl")

['model_columns.pkl']

In [47]:
threshold = 0.4
joblib.dump(threshold, "threshold.pkl")

['threshold.pkl']

In [48]:
def predict_student(input_df):
    model = joblib.load("student_risk_model.pkl")
    columns = joblib.load("model_columns.pkl")
    threshold = joblib.load("threshold.pkl")

    input_encoded = pd.get_dummies(input_df)
    input_encoded = input_encoded.reindex(columns=columns, fill_value=0)

    prob = model.predict_proba(input_encoded)[:,1][0]
    prediction = 1 if prob > threshold else 0

    return prob, prediction